# Training the Direct Approach Classifier

## Overview

This notebook trains a MobileNet-based classifier using the **DIRECT** approach from Section 5 of "Feature-Driven Priority Queuing" (Appendix B). Instead of predicting disease probabilities and then mapping to queues via a routing matrix, the direct approach:

1. Attaches a softmax layer to MobileNet that outputs **N=4 queue assignment probabilities** directly
2. Trains with a custom **queuing loss** (Eq. 27) that minimizes empirical waiting cost
3. Tests 4 delay-cost structures:
   - $c_i = 1.8(T-i)+1$ (linear, moderate slope)
   - $c_i = 10(T-i)+1$ (linear, steep slope)
   - $c_i = 1.5^{T-i}$ (convex, moderate base)
   - $c_i = 1.8^{T-i}$ (convex, steep base)

All with $\rho=0.9$ (server utilization) and $T=14$ image types.

### Mathematical Background

The queuing loss minimizes the **empirical average waiting cost** (Eq. 27):

$$L_q(\hat{P}^d) = \sum_{n=1}^{N} \frac{(c \cdot \lambda)_n \, E[R]}{(1 - \sigma_n)(1 - \sigma_{n-1})}$$

where:
- $\hat{P}$ is the empirical classification matrix ($T \times N$) computed from batch predictions
- $(c \cdot \lambda)_n = \sum_{i=1}^{T} c_i \lambda_i P_{i,n}$ (cost-weighted arrivals for queue $n$)
- $\sigma_n = \sum_{k=1}^{n} \bar{\rho}_k$ is the **cumulative** utilization through queue $n$, with $\bar{\rho}_k = \sum_{i=1}^{T} \rho_i P_{i,k}$ and $\sigma_0 = 0$
- $E[R]$ is the mean residual service time

## Section 1: Setup & Configuration

In [ ]:
import numpy as np
import pandas as pd
import os
from glob import glob
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
import pickle
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════

# Directories
DATA_DIR = '../data'
WEIGHTS_DIR = '../model_weights'
LOGS_DIR = 'logs'

# Create directories if they don't exist
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

# Image processing
IMG_SIZE = (512, 512)
COLOR_MODE = 'rgb'
CHANNELS = 3

# Training hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 0.0005
STEPS_PER_EPOCH = 1000
EPOCHS = 100
MIN_CASES = 1000
RANDOM_STATE = 2018

# Queueing system parameters (from Section 5 of the paper)
T = 14    # Number of image types (diseases + no finding)
N = 4     # Number of priority queues
RHO = 0.90  # Server utilization (traffic intensity threshold)

# ──────────────────────────────────────────────────────────────────────────────
# DELAY-COST STRUCTURES (Section 5.1)
# ──────────────────────────────────────────────────────────────────────────────

COST_STRUCTURES = {
    'linear_1.8':  {
        'label': r'$c_i = 1.8(T-i)+1$',
        'func': lambda i, t=T: 1.8 * (t - i) + 1
    },
    'linear_10':   {
        'label': r'$c_i = 10(T-i)+1$',
        'func': lambda i, t=T: 10 * (t - i) + 1
    },
    'convex_1.5':  {
        'label': r'$c_i = 1.5^{T-i}$',
        'func': lambda i, t=T: 1.5 ** (t - i)
    },
    'convex_1.8':  {
        'label': r'$c_i = 1.8^{T-i}$',
        'func': lambda i, t=T: 1.8 ** (t - i)
    },
}

print(f"Configuration:")
print(f"  - Image types (T): {T}")
print(f"  - Priority queues (N): {N}")
print(f"  - Server utilization (ρ): {RHO}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"\nCost structures to train:")
for key, info in COST_STRUCTURES.items():
    print(f"  - {key}: {info['label']}")

## Section 2: Data Loading & Preprocessing

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DISEASE RANKING (Table 2)
# ══════════════════════════════════════════════════════════════════════════════

DISEASE_RANKING = [
    'Pneumothorax',       # Type 1: Most critical
    'Emphysema',          # Type 2
    'Pneumonia',          # Type 3
    'Edema',              # Type 4
    'Consolidation',      # Type 5
    'Effusion',           # Type 6
    'Infiltration',       # Type 7
    'Atelectasis',        # Type 8
    'Cardiomegaly',       # Type 9
    'Pleural_Thickening', # Type 10
    'Fibrosis',           # Type 11
    'Mass',               # Type 12
    'Nodule',             # Type 13
    'No Finding'          # Type 14: Least critical (or healthy)
]

print(f"Disease ranking (Table 2):")
for idx, disease in enumerate(DISEASE_RANKING, 1):
    print(f"  Type {idx:2d}: {disease}")

In [ ]:
def image_type(finding_labels):
    """
    Map finding labels to image types 1-14 based on disease ranking (Table 2).
    
    If a finding appears in the ranked disease list, return its type (index+1).
    Otherwise, assign to type 14 (No Finding / lowest priority).
    
    Args:
        finding_labels: List of disease findings (strings)
    
    Returns:
        int: Type index (1-14)
    """
    for label in finding_labels:
        if label in DISEASE_RANKING:
            return DISEASE_RANKING.index(label) + 1  # 1-indexed
    # If no recognized disease, assign to type 14 (No Finding)
    return T

# Test the function
print("Testing image_type function:")
print(f"  ['Pneumothorax'] -> Type {image_type(['Pneumothorax'])}")
print(f"  ['Nodule', 'Mass'] -> Type {image_type(['Nodule', 'Mass'])}")
print(f"  ['No Finding'] -> Type {image_type(['No Finding'])}")
print(f"  [] -> Type {image_type([])}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD ChestX-ray14 DATA
# ══════════════════════════════════════════════════════════════════════════════

# Load the preprocessed data (from Section 01_Prepare_Data)
data_path = os.path.join(DATA_DIR, 'data.pkl')
assert os.path.exists(data_path), f"Data file not found: {data_path}"

with open(data_path, 'rb') as f:
    data = pickle.load(f)

print(f"Loaded data: {len(data)} records")
print(f"Columns: {data.columns.tolist()}")
print(f"\nFirst few records:")
print(data.head())

In [ ]:
# Filter for cases with at least MIN_CASES samples
disease_counts = pd.Series()
for idx, row in data.iterrows():
    findings = [col for col in data.columns if col.startswith('Finding_')]
    labels = [col.replace('Finding_', '') for col in findings if row[col] == 1]
    if labels:
        for label in labels:
            disease_counts[label] = disease_counts.get(label, 0) + 1

print(f"Disease counts in full dataset:")
print(disease_counts.sort_values(ascending=False))
print(f"\nTotal unique diseases: {len(disease_counts)}")

In [ ]:
# Add Type column based on findings
def get_type_from_row(row):
    findings = [col.replace('Finding_', '') for col in data.columns 
                if col.startswith('Finding_') and row[col] == 1]
    return image_type(findings)

data['Type'] = data.apply(get_type_from_row, axis=1)

print(f"Type distribution:")
type_counts = data['Type'].value_counts().sort_index()
for type_idx, count in type_counts.items():
    disease = DISEASE_RANKING[type_idx - 1]
    pct = 100 * count / len(data)
    print(f"  Type {type_idx:2d} ({disease:20s}): {count:6d} ({pct:5.1f}%)")

## Section 3: Train/Test Split & Balancing

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 50/25/25 STRATIFIED SPLIT
# ══════════════════════════════════════════════════════════════════════════════

# Split: 50% train, 25% val, 25% test (stratified by Type)
X_train, X_temp, y_train, y_temp = train_test_split(
    data[['path']],
    data['Type'],
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=data['Type']
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# Reconstruct dataframes with Type column
train_data = X_train.copy()
train_data['Type'] = y_train.values

val_data = X_val.copy()
val_data['Type'] = y_val.values

test_data = X_test.copy()
test_data['Type'] = y_test.values

print(f"Train set: {len(train_data)} samples (50%)")
print(f"Validation set: {len(val_data)} samples (25%)")
print(f"Test set: {len(test_data)} samples (25%)")
print(f"\nTrain set type distribution:")
print(train_data['Type'].value_counts().sort_index())

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# OVERSAMPLING RARE CLASSES (using RandomOverSampler)
# ──────────────────────────────────────────────────────────────────────────────

# Reshape for sklearn compatibility
X_train_array = train_data.reset_index(drop=True)['path'].values.reshape(-1, 1)
y_train_array = train_data['Type'].values

# Apply oversampling
oversampler = RandomOverSampler(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = oversampler.fit_resample(X_train_array, y_train_array)

# Reconstruct dataframe
train_data_balanced = pd.DataFrame({
    'path': X_train_resampled[:, 0],
    'Type': y_train_resampled
})

print(f"After oversampling: {len(train_data_balanced)} samples")
print(f"Balanced type distribution:")
print(train_data_balanced['Type'].value_counts().sort_index())

## Section 4: Image Data Generator

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA AUGMENTATION
# ══════════════════════════════════════════════════════════════════════════════

# Training augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation/test: only rescaling
val_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

print("Image data generators configured:")
print(f"  - Training: rescaling + rotation + shift + zoom + flip")
print(f"  - Validation: rescaling only")
print(f"  - Test: rescaling only")

In [ ]:
# Create data generators from dataframes
# IMPORTANT: For the direct approach, we use class_mode='categorical' because:
#   - y_true will be one-hot encoded type labels (T=14 classes)
#   - y_pred will be queue probabilities (N=4 outputs)
#   - The queue_loss_tensor function maps between these

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_data_balanced.reset_index(drop=True),
    x_col='path',
    y_col='Type',
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    classes=list(range(1, T + 1)),
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_STATE
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_data.reset_index(drop=True),
    x_col='path',
    y_col='Type',
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    classes=list(range(1, T + 1)),
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_data.reset_index(drop=True),
    x_col='path',
    y_col='Type',
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    classes=list(range(1, T + 1)),
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Data generators created:")
print(f"  - Training: {len(train_data_balanced)} samples")
print(f"  - Validation: {len(val_data)} samples")
print(f"  - Test: {len(test_data)} samples")

## Section 5: Queuing Cost Functions

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS FOR QUEUING PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════

def get_costs(cost_key):
    """
    Compute delay costs for all T types given a cost structure.
    
    Args:
        cost_key: Key in COST_STRUCTURES dict
    
    Returns:
        np.array: T-vector of costs [c_1, c_2, ..., c_T]
    """
    func = COST_STRUCTURES[cost_key]['func']
    return np.array([func(i) for i in range(1, T + 1)], dtype=np.float32)


def get_arrival_rates(data_df):
    """
    Estimate arrival rates from type frequencies in the dataset.
    
    Args:
        data_df: DataFrame with 'Type' column
    
    Returns:
        np.array: T-vector of arrival rates [λ_1, λ_2, ..., λ_T]
                  Normalized so they sum to 1
    """
    type_counts = data_df['Type'].value_counts().sort_index()
    # Ensure all types are represented (add zeros for missing types)
    arrival_rates = np.zeros(T, dtype=np.float32)
    for type_idx, count in type_counts.items():
        arrival_rates[type_idx - 1] = count
    return arrival_rates / arrival_rates.sum()


def get_service_rates():
    """
    Compute service rates assuming identical service times across all types.
    
    Under the assumption of equal service times (µ_i = µ for all i),
    the traffic intensity for each type is ρ_i = λ_i / µ.
    To match the target utilization ρ, we scale service rates appropriately.
    
    Returns:
        np.array: T-vector of service rates [μ_1, μ_2, ..., μ_T]
    """
    # For simplicity, set service rate = 1/ρ for all types
    # This ensures total system utilization = ρ
    return np.array([1.0 / RHO] * T, dtype=np.float32)


# Test these functions
print("Testing cost functions:")
costs_linear = get_costs('linear_1.8')
print(f"  linear_1.8 costs: {costs_linear}")

arrivals = get_arrival_rates(train_data)
print(f"\nArrival rates (training data):")
print(f"  Sum: {arrivals.sum():.4f}")
print(f"  Mean: {arrivals.mean():.4f}")

service = get_service_rates()
print(f"\nService rates:")
print(f"  All equal: {np.allclose(service, service[0])}")
print(f"  Value: {service[0]:.4f}")

In [ ]:
def queue_wait_cost(pmat, costs, arrivals, service):
    """
    Compute average waiting cost given a classification matrix P.
    
    This implements Equation 3 from the paper: the steady-state average
    waiting time in an M/G/1 non-preemptive priority queue system with N parallel queues,
    weighted by delay costs.
    
    Args:
        pmat: T x N classification matrix (P[i,n] = probability type i goes to queue n)
        costs: T-vector of delay costs [c_1, ..., c_T]
        arrivals: T-vector of arrival rates [λ_1, ..., λ_T], should sum to 1
        service: T-vector of service rates [μ_1, ..., μ_T]
    
    Returns:
        float: Average waiting cost C(P)
    """
    # Weighted arrivals and traffic intensity by type
    clambda = costs * arrivals  # Element-wise: c_i * λ_i
    rho_vec = arrivals / service  # Element-wise: ρ_i = λ_i / μ_i
    
    # Aggregate by queue
    clambda_per_queue = np.matmul(clambda, pmat)  # Shape: (N,)
    rho_per_queue = np.matmul(rho_vec, pmat)      # Shape: (N,)
    
    # Cumulative traffic intensity
    rho_cumsum = np.cumsum(rho_per_queue)  # Σ_{n'=1}^{n} ρ_{n'}
    
    # Denominators: (1 - ρ_n)(1 - ρ_{n-1})
    vec1 = 1 - rho_cumsum
    vec2 = np.roll(1 - rho_cumsum, 1)
    vec2[0] = 1  # ρ_0 = 0
    
    # Second moment of service time
    E_R = np.sum(arrivals / (service ** 2))
    
    # Waiting cost (Eq. 3)
    with np.errstate(divide='ignore', invalid='ignore'):
        wait = np.sum(clambda_per_queue * E_R / (vec1 * vec2 + 1e-10))
    
    return float(wait)


# Test with identity matrix (all types to all queues equally)
pmat_test = np.ones((T, N)) / N
costs_test = get_costs('linear_1.8')
arrivals_test = get_arrival_rates(train_data)
service_test = get_service_rates()

wait_cost_test = queue_wait_cost(pmat_test, costs_test, arrivals_test, service_test)
print(f"\nTest: uniform routing matrix, wait cost = {wait_cost_test:.6f}")

## Section 6: Custom Queuing Loss Function

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CUSTOM QUEUING LOSS (Eq. 27 in Paper)
# ══════════════════════════════════════════════════════════════════════════════

# Global variables that will be set before training each model
# These are captured by the loss function closure
global_costs = None
global_arrivals = None
global_service = None


def queue_loss_tensor(y_true, y_pred):
    """
    Custom TensorFlow loss: empirical queuing cost C(P_hat^d) from Eq. 27.
    
    This loss function minimizes the empirical average waiting cost in the queue
    system, computed from the batch's assignment of image types to queues.
    
    Args:
        y_true: (batch_size, T) one-hot encoded type labels
                y_true[b, i] = 1 if sample b is type i, else 0
        y_pred: (batch_size, N) queue assignment probabilities
                y_pred[b, n] = P(sample b assigned to queue n)
    
    Returns:
        Scalar loss value (average waiting cost)
    
    Key insight: The loss computes an empirical classification matrix P_hat that
    describes how the batch's types are distributed across queues, then evaluates
    the steady-state waiting cost for that distribution.
    """
    # Cast to float16 for numerical stability during training
    y_true = tf.cast(y_true, tf.float16)
    y_pred = tf.cast(y_pred, tf.float16)
    
    # Empirical classification matrix P_hat = (1/n_i) * (# type i samples) @ y_pred
    # Shape: y_true^T (T x batch_size) @ y_pred (batch_size x N) = (T x N)
    pmat = tf.matmul(tf.transpose(y_true), y_pred)
    
    # Normalize each row by the count of that type in the batch
    # Shape: (T, 1)
    type_counts = tf.reshape(tf.reduce_sum(y_true, axis=0), [-1, 1])
    pmat = pmat / (type_counts + 1e-10)
    
    # Convert global parameters to tensors
    costs_t = tf.constant(global_costs, dtype=tf.float16)
    arrivals_t = tf.constant(global_arrivals, dtype=tf.float16)
    service_t = tf.constant(global_service, dtype=tf.float16)
    
    # Weighted arrivals and traffic intensity
    clambda = costs_t * arrivals_t  # Shape: (T,)
    rho = arrivals_t / service_t      # Shape: (T,)
    
    # Aggregate by queue
    clambda_per_queue = tf.linalg.matvec(pmat, clambda, transpose_a=True)  # Shape: (N,)
    rho_per_queue = tf.linalg.matvec(pmat, rho, transpose_a=True)           # Shape: (N,)
    
    # Cumulative traffic intensity: ρ_1, ρ_1+ρ_2, ..., ρ_1+...+ρ_N
    rho_cumsum = tf.cumsum(rho_per_queue)
    
    # Denominators: (1 - ρ_n) and (1 - ρ_{n-1})
    vec1 = 1.0 - rho_cumsum
    vec2 = tf.concat([[1.0], 1.0 - tf.cumsum(rho_per_queue)[:-1]], axis=0)
    
    # Second moment of service times
    E_R = tf.reduce_sum(arrivals_t / (service_t ** 2))
    
    # Waiting cost (Eq. 27 in paper)
    wait = tf.reduce_sum(
        clambda_per_queue * E_R / (vec1 * vec2 + 1e-10)
    )
    
    return wait


print("Queue loss function defined.")
print("Note: Global variables global_costs, global_arrivals, global_service")
print("      must be set before compiling the model.")

## Section 7: Model Architecture

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL ARCHITECTURE
# ══════════════════════════════════════════════════════════════════════════════

def build_direct_model(img_size, channels, n_queues):
    """
    Build MobileNet-based classifier for direct queue assignment.
    
    Architecture:
      - Input: RGB image (img_size[0] x img_size[1] x channels)
      - MobileNet base (pre-trained weights): extracts features
      - GlobalAveragePooling2D: reduces spatial dimensions
      - Dense(512) + Dropout(0.5): classification head
      - Dense(512) + Dropout(0.5): additional capacity
      - Dense(n_queues, softmax): direct queue assignment
    
    Output:
      - N-dimensional softmax vector of queue assignment probabilities
      - These are combined with type labels via queue_loss_tensor
    
    Args:
        img_size: (height, width) tuple
        channels: number of color channels (3 for RGB)
        n_queues: number of priority queues (N=4)
    
    Returns:
        Compiled Keras Sequential model
    """
    model = Sequential()
    
    # MobileNet base
    base_model = MobileNet(
        input_shape=(*img_size, channels),
        include_top=False,
        weights=None  # Random initialization (no pre-training)
    )
    model.add(base_model)
    
    # Feature aggregation
    model.add(GlobalAveragePooling2D())
    
    # Classification head
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.5))
    
    # Queue assignment layer (softmax over N queues)
    model.add(Dense(n_queues, activation='softmax', name='queue_output'))
    
    return model


# Build a test model to check architecture
test_model = build_direct_model(IMG_SIZE, CHANNELS, N)
test_model.summary()

## Section 8: Training Loop

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP: Train one model per cost structure
# ══════════════════════════════════════════════════════════════════════════════

training_history = {}

for cost_key, cost_info in COST_STRUCTURES.items():
    print(f"\n" + "="*80)
    print(f"Training with cost structure: {cost_key}")
    print(f"  Label: {cost_info['label']}")
    print(f"="*80)
    
    # Set global queuing parameters for this cost structure
    global_costs = get_costs(cost_key)
    global_arrivals = get_arrival_rates(train_data)
    global_service = get_service_rates()
    
    print(f"Queuing parameters:")
    print(f"  Costs (min/max): {global_costs.min():.4f} / {global_costs.max():.4f}")
    print(f"  Arrivals (sum): {global_arrivals.sum():.4f}")
    print(f"  Service (constant): {global_service[0]:.4f}")
    
    # Build model
    model = build_direct_model(IMG_SIZE, CHANNELS, N)
    
    # Compile with custom queue loss
    optimizer = optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer=optimizer,
        loss=queue_loss_tensor,
        metrics=['accuracy']  # Standard accuracy metric (for monitoring)
    )
    
    print(f"\nModel compiled with:")
    print(f"  Optimizer: Adam (lr={LEARNING_RATE})")
    print(f"  Loss: custom queue_loss_tensor")
    
    # Callbacks
    checkpoint = callbacks.ModelCheckpoint(
        filepath=os.path.join(WEIGHTS_DIR, f'direct_{cost_key}_rho_{RHO:.2f}_best.hdf5'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
    
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        verbose=1,
        restore_best_weights=True
    )
    
    tensorboard = callbacks.TensorBoard(
        log_dir=os.path.join(LOGS_DIR, f'direct_{cost_key}'),
        histogram_freq=0
    )
    
    # Train
    print(f"\nTraining...")
    history = model.fit(
        train_generator,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS,
        validation_data=val_generator,
        callbacks=[checkpoint, early_stop, tensorboard],
        verbose=1
    )
    
    training_history[cost_key] = history
    
    # Save final weights
    final_weights_path = os.path.join(WEIGHTS_DIR, f'direct_{cost_key}_rho_{RHO:.2f}_final.hdf5')
    model.save_weights(final_weights_path)
    print(f"\nFinal weights saved to: {final_weights_path}")
    
    print(f"\nTraining complete for {cost_key}")

## Section 9: Training Results Summary

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING RESULTS VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (cost_key, history) in enumerate(training_history.items()):
    ax = axes[idx]
    
    ax.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Queuing Loss', fontsize=11)
    ax.set_title(f'{cost_key}\n{COST_STRUCTURES[cost_key]["label"]}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Add final loss values
    final_train = history.history['loss'][-1]
    final_val = history.history['val_loss'][-1]
    ax.text(0.98, 0.05, f'Final:\nTrain: {final_train:.6f}\nVal: {final_val:.6f}',
            transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, 'training_curves_direct.png'), dpi=150, bbox_inches='tight')
print("Training curves saved to: logs/training_curves_direct.png")
plt.show()

In [ ]:
# Print summary statistics
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)

for cost_key, history in training_history.items():
    print(f"\n{cost_key} ({COST_STRUCTURES[cost_key]['label']}):")
    print(f"  Epochs trained: {len(history.history['loss'])}")
    print(f"  Initial train loss: {history.history['loss'][0]:.6f}")
    print(f"  Final train loss: {history.history['loss'][-1]:.6f}")
    print(f"  Best val loss: {min(history.history['val_loss']):.6f}")
    print(f"  Final val loss: {history.history['val_loss'][-1]:.6f}")
    
    # Improvement
    improvement = 100 * (history.history['loss'][0] - history.history['loss'][-1]) / history.history['loss'][0]
    print(f"  Loss improvement: {improvement:.1f}%")

## Section 10: Model Evaluation on Test Set

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION ON TEST SET
# ══════════════════════════════════════════════════════════════════════════════

test_results = {}

for cost_key in COST_STRUCTURES.keys():
    print(f"\nEvaluating {cost_key}...")
    
    # Set global parameters for evaluation
    global_costs = get_costs(cost_key)
    global_arrivals = get_arrival_rates(train_data)  # Use training distribution
    global_service = get_service_rates()
    
    # Load best weights
    weights_path = os.path.join(WEIGHTS_DIR, f'direct_{cost_key}_rho_{RHO:.2f}_best.hdf5')
    if not os.path.exists(weights_path):
        weights_path = os.path.join(WEIGHTS_DIR, f'direct_{cost_key}_rho_{RHO:.2f}_final.hdf5')
    
    model = build_direct_model(IMG_SIZE, CHANNELS, N)
    model.compile(optimizer='adam', loss=queue_loss_tensor, metrics=['accuracy'])
    model.load_weights(weights_path)
    
    # Evaluate
    test_loss, test_acc = model.evaluate(test_generator, verbose=0)
    test_results[cost_key] = {
        'loss': test_loss,
        'accuracy': test_acc
    }
    
    print(f"  Test loss: {test_loss:.6f}")
    print(f"  Test accuracy: {test_acc:.4f}")

In [ ]:
# Plot test results comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

cost_keys = list(test_results.keys())
losses = [test_results[k]['loss'] for k in cost_keys]
accuracies = [test_results[k]['accuracy'] for k in cost_keys]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Loss comparison
ax1.bar(range(len(cost_keys)), losses, color=colors, alpha=0.7)
ax1.set_xticks(range(len(cost_keys)))
ax1.set_xticklabels([k.replace('_', '\n') for k in cost_keys], fontsize=10)
ax1.set_ylabel('Test Loss (Queuing Cost)', fontsize=11)
ax1.set_title('Test Loss by Cost Structure', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, loss in enumerate(losses):
    ax1.text(i, loss + 0.01, f'{loss:.4f}', ha='center', fontsize=9)

# Accuracy comparison
ax2.bar(range(len(cost_keys)), accuracies, color=colors, alpha=0.7)
ax2.set_xticks(range(len(cost_keys)))
ax2.set_xticklabels([k.replace('_', '\n') for k in cost_keys], fontsize=10)
ax2.set_ylabel('Test Accuracy', fontsize=11)
ax2.set_title('Test Accuracy by Cost Structure', fontsize=12, fontweight='bold')
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, acc in enumerate(accuracies):
    ax2.text(i, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, 'test_results_direct.png'), dpi=150, bbox_inches='tight')
print("Test results saved to: logs/test_results_direct.png")
plt.show()

## Section 11: Save All Models and Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE TRAINING METADATA
# ══════════════════════════════════════════════════════════════════════════════

# Save configuration
config = {
    'approach': 'direct',
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'epochs': EPOCHS,
    'T': T,
    'N': N,
    'RHO': RHO,
    'cost_structures': list(COST_STRUCTURES.keys()),
    'disease_ranking': DISEASE_RANKING,
}

config_path = os.path.join(WEIGHTS_DIR, 'config_direct.pkl')
with open(config_path, 'wb') as f:
    pickle.dump(config, f)

print(f"Configuration saved to: {config_path}")

# Save test results
results_path = os.path.join(LOGS_DIR, 'test_results_direct.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(test_results, f)

print(f"Test results saved to: {results_path}")

# List all saved files
print(f"\nSaved weights:")
for fname in sorted(os.listdir(WEIGHTS_DIR)):
    if 'direct' in fname:
        fpath = os.path.join(WEIGHTS_DIR, fname)
        fsize = os.path.getsize(fpath) / (1024**2)  # Convert to MB
        print(f"  - {fname} ({fsize:.1f} MB)")

In [ ]:
print("\n" + "="*80)
print("DIRECT APPROACH TRAINING COMPLETE")
print("="*80)
print(f"\nKey Results:")
print(f"  - 4 models trained (one per cost structure)")
print(f"  - Queueing system: T={T} types, N={N} queues, ρ={RHO}")
print(f"  - Custom loss: queue_loss_tensor (Eq. 27 in paper)")
print(f"\nCost structures:")
for cost_key in COST_STRUCTURES.keys():
    print(f"  - {cost_key}: {COST_STRUCTURES[cost_key]['label']}")
print(f"\nTest Set Performance:")
for cost_key, results in test_results.items():
    print(f"  - {cost_key}:")
    print(f"      Loss: {results['loss']:.6f}")
    print(f"      Accuracy: {results['accuracy']:.4f}")
print(f"\nWeights and logs saved to:")
print(f"  - {WEIGHTS_DIR}/")
print(f"  - {LOGS_DIR}/")